# NB01: Data Collection

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250104200

</div>

## Setup

Import required packages

In [3]:
import os
import requests
import json
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path


Get an API key from [US Bureau of Labour Statistics API registration](https://data.bls.gov/registrationEngine/), and save it in the `.env` file in this folder.

The cell below reads that file and loads the key into a variable.

In [4]:
load_dotenv()

BLS_API_KEY = os.getenv("BLS_API_KEY")

## Request employment data from BLS

Call the BLS API for employment data by industry at the 4-digit NAICS code level.




Compile the list of series IDs required following this structure:

	Series ID    CEU0800000003
	Positions       Value           Field Name
	1-2             CE              Prefix
	3               U               Seasonal Adjustment Code
	4-11		08000000	Supersector and Industry Codes
	12-13           03              Data Type Code

The industry codes need to be unpacked from the [codes list](https://download.bls.gov/pub/time.series/ce/ce.industry) provided by the BLS. This is contained in a `.tsv` file but is hosted with a faulty extension, so it must be downloaded manually and read as a .tsv.

Some industry codes map onto multiple NAICS codes or vice versa, reflected in the `naics_code` column with either a comma separating the additional final digits, or the NAICS code being split as "part XXXX" across two industry codes.

In order to preserve a one to one mapping to exposure scores I drop these codes.

In [5]:
industry_code_map = (
    pd.read_csv("../data/reference/ce_industry.tsv", delimiter="\t", index_col=False)
    # Selecting display level 5 filters on 4 digit NAICS codes to match with AIOE table
    .query("display_level == 5")
    .filter(["industry_code", "naics_code", "industry_name"])
    .reset_index(drop=True)
)

# Filter out codes without a 1:1 industry code mapping
industry_code_map = industry_code_map[industry_code_map["naics_code"].str.len() == 4]

Define function to generate BLS series ID and constant parameters for analysis:

* Time period: 2019 to 2026
* Seasonal adjustment:
* Data variable: Total employment in 1000s

In [6]:
SERIES = "CE" # National Employment, Hours, and Earnings Survey identifier
SA = "S" # Seasonally adjusted data
data_type = "01" # Employement (Level 1000s)

START_YEAR = 2019
END_YEAR = 2026

def generate_emp_series_id(industry, series=SERIES, seas_adj=SA, d_type=data_type):
    return f"{series}{seas_adj}{industry}{d_type}"


Add column of BLS series IDs to `industry_code_map` by applying `generate_series_id function`

In [8]:
employment_map = industry_code_map.assign(
    series_id = [generate_emp_series_id(x) for x in industry_code_map["industry_code"]]
    )
employment_map.reset_index(drop=True)

,industry_code,naics_code,industry_name,series_id
0,10113300,1133,Logging,CES1011330001
1,10212100,2121,Coal mining,CES1021210001
2,10212200,2122,Metal ore mining,CES1021220001
3,10212300,2123,Nonmetallic mineral mining and quarrying,CES1021230001
4,20236100,2361,Residential building construction,CES2023610001
...,...,...,...,...
211,80812900,8129,Other personal services,CES8081290001
212,80813200,8132,Grantmaking and giving services,CES8081320001
213,80813300,8133,Social advocacy organizations,CES8081330001
214,80813400,8134,Civic and social organizations,CES8081340001


In [9]:
def bls_api_call(
    series_ids: list,
    api_key=BLS_API_KEY,
    start_year=START_YEAR,
    end_year=END_YEAR,
    timeout = 60
    ):
    ''' 
    Requests data from BLS API from a list of up to 50 BLS series IDs.
    Returns the response in JSON format
    '''
    # headers required by BLS API
    headers = {'Content-type': 'application/json'}
    # Generate request JSON to post to BLS API
    request_data = json.dumps({
            "seriesid":series_ids, 
            "startyear":START_YEAR, 
            "endyear":END_YEAR,
            "registrationkey":BLS_API_KEY
            }
            )
    # Post query to BLS API
    p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', data=request_data, headers=headers)
    return p.json()


Function to save raw JSON data pulled from BLS API.

In [18]:
def save_raw_data(data, indicator, batch):
    '''
    Saves raw JSON returned by BLS API call into the raw data folder.
    The data will be stored in a folder with the name of the indicator,
    and a batch number will be attached to the end of the file name.
    '''
    folder = Path("..") / "data" / "raw" / indicator.lower()
    
    os.makedirs(folder, exist_ok=True)

    file_path = folder / f"bls_{indicator.lower()}_{batch}.json"

    with open(file_path, mode="w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    print(f"Saved to {file_path}")

The BLS API will return up to 50 series IDs per call, so I define a function that will take a long list of series IDs and put them into a list containing batches of 50 series IDs.

In [1]:
def batch_series_ids(series_ids, batch_size=50):
    '''
    Takes a list of series IDs and places them into batches of 50.
    Returns a list containing lists of the batched IDs and prints
    the total number of IDs batched and the number of batches generated 
    as a validation check.
    Different batch sizes can be chosen with parameter batch_size.
    '''
    batch_list = []
    num_series_ids = len(series_ids)
    start = 0
    end = batch_size

    for i in range(num_series_ids // batch_size):
        batch = series_ids[start:end]
        batch_list.append(batch)
        start += batch_size
        end += batch_size
    
    final_batch = num_series_ids - num_series_ids % batch_size
    batch_list.append(
        series_ids[final_batch+1:]
    )

    # check total number of series IDs and batches
    total_batched = sum([len(i) for i in batch_list])
    print(f"{total_batched} series IDs placed into {len(batch_list)} batches")

    return batch_list

emp_batches = batch_series_ids(industry_code_map["series_id"].tolist())

emp_batches

NameError: name 'industry_code_map' is not defined

Call API and save raw data

In [19]:
raw_json = bls_api_call(["CES1021220001"])

save_raw_data(raw_json, indicator="employment", batch=1)

Saved to ../data/raw/employment/bls_employment_1.json


In [1]:
# Temporary DF conversion to test raw data collection
df = pd.json_normalize(
    json_data["Results"]["series"],
    record_path="data")
df

NameError: name 'pd' is not defined